# Practice 3 — Hugging Face: Sentiment Analysis & Fine-tuning

**Report notebook (read-only):** loads locally produced artifacts. It **does not download a model, execute training or generate fabricated results**. Run scripts first.

**Student:** [fill in] · **Date:** [fill in] · **Run:** `lan_1` (edit below).


## 1. Aim and acceptance checklist

**Exercise 1:** install Transformers, pretrained Hub sentiment classifier, tokenization and prediction. **Exercise 2:** install Transformers/Datasets/Evaluate, IMDb data, tokenizer/model, preprocessing, TrainingArguments, Trainer fine-tuning and final evaluation.

Data split: train/validation from official IMDb train, reserved test from official IMDb test. See `Experiment_Plan.md` and `docs/`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    candidate = ROOT / "practice_3_huggingface"
    if candidate.exists(): ROOT = candidate
RUN = 'lan_1'  # change to your completed run
run = ROOT / 'outputs' / 'runs' / RUN

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None

print('Practice root:', ROOT.resolve())
print('Run:', RUN, '| Complete:', (run / 'TRAINING_COMPLETE').exists())


## 2. Exercise 1 — model, tokens, inference

A pretrained sentiment classifier performs **inference**; it does not update weights. Show exact input, tokens, token IDs and model response. Scores are model predictions, not calibrated guarantees.

In [ ]:
ex1 = read_json(ROOT / 'outputs' / 'exercise1' / 'sentiment_result.json')
if ex1 is None:
    print('Exercise 1 artifact absent. Run src/1_sentiment.py first.')
else:
    for key in ['model_id','sentence','tokens','token_ids','predicted_label','model_score']:
        print(f'{key}: {ex1.get(key)}')


## 3. Dataset, labels and split integrity

The only labeled classes are 0=NEGATIVE and 1=POSITIVE. Report selected subset counts and SHA256; test is not used for checkpoint selection.

In [ ]:
split = read_json(ROOT / 'outputs' / 'data' / 'split_info.json')
if split is None:
    print('No prepared data. Run src/2_preparedata.py.')
else:
    print('Dataset:', split.get('dataset_id'))
    print('Seed:', split.get('seed'), 'Max length:', split.get('max_length'))
    display(pd.DataFrame([{'split':k, **v['labels'], 'n':v['count'], 'sha256':v['text_label_sha256'][:16]+'…'}
                          for k,v in split['splits'].items()]))


## 4. Fine-tuning configuration and selected run

Fine-tune `distilbert-base-uncased` for binary classification. A new head emits two logits per review, and Trainer handles forward, cross entropy, backward and optimizer steps.

In [ ]:
cfg = read_json(run / 'config.json')
if cfg is None:
    print('No configuration found. Training may not have started.')
else:
    for k in ('base_model','dataset_id','seed','train_samples','validation_samples','max_length',
              'epochs','batch_size','learning_rate','weight_decay','best_checkpoint','best_metric'):
        print(f'{k}: {cfg.get(k, "not recorded")}')


## 5. Train and validation evidence

Read the actual Trainer history and run summaries. Empty outputs mean **not measured**, not zero. Differentiate training and validation losses.

In [ ]:
history = read_json(run / 'history.json')
if history is None:
    print('history.json absent. No completed training history to report.')
else:
    interesting = ['epoch','step','loss','eval_loss','eval_accuracy','eval_precision','eval_recall','eval_f1','learning_rate']
    table = pd.DataFrame(history)
    display(table[[c for c in interesting if c in table.columns]].tail(20))
    print('Actual history rows:',len(table))
print('Training:', read_json(run / 'train_metrics.json'))
print('Validation:', read_json(run / 'validation_metrics.json'))


## 6. Final independent evaluation

`validation_recheck.json` is not the same as `test_metrics.json`. The final test should be run once after selection is frozen, and evidence must identify the split.

In [ ]:
test = read_json(run / 'test_metrics.json')
val = read_json(run / 'validation_recheck.json')
evidence = test if test is not None else val
if evidence is None:
    print('No prediction-level evaluation. Run 4_evaluate.py on validation first.')
else:
    print('Split:', evidence.get('split'), '| count:', evidence.get('sample_count'))
    display(pd.Series(evidence.get('metrics', {}),name='metric').to_frame())
    print('ROC-AUC:',evidence.get('roc_auc'))
    if evidence.get('classification_report'):
        display(pd.DataFrame(evidence['classification_report']).T)
    print('Confusion matrix (rows actual; columns predicted):',evidence.get('confusion_matrix'))


## 7. Generated figures

Render `loss.png`, `validation_scores.png`, and `confusion_matrix.png` only when `5_report.py` has created them from real artifacts.

In [ ]:
images = ROOT / 'visualization' / RUN
for name in ['loss.png','validation_scores.png','confusion_matrix.png']:
    p = images / name
    if p.exists():
        display(Markdown('**'+name+'**'))
        display(Image(filename=str(p)))
    else: print('Not generated:',name)


## 8. Interpretation and limitations

Complete this part **after reviewing real metrics**:

1. Did validation F1 improve and is the model under-/overfitting?
2. Which class has lower recall and what are FN/FP?
3. What does token truncation (max 128) mean for long reviews?
4. Were hyperparameters chosen without inspecting the final test?
5. Can results from a 2,000-sample, single-seed run generalize to full IMDb or another domain? **Not necessarily.**

**Conclusion:** [add grounded quantitative summary citing `test_metrics.json`].